# Pipeline dos Dados

In [56]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [57]:
PROCESSED = Path('../../data/processed')
df_edges = pd.read_csv(PROCESSED / 'edges.csv')

RAW = Path('../../data/raw')
df_casos = pd.read_csv(RAW / 'cases.csv')

print(f"Dados reais carregados! Total de arestas com contexto: {len(df_edges)}")
display(df_edges.head())

print(f"Dados de casos carregados! Total de casos: {len(df_casos)}")
display(df_casos.head())

Dados reais carregados! Total de arestas com contexto: 654


,edge_id,case_id,source_label,target_label,relation,contexto
0,E_001,PMC10106591_01,female,intermittent chest pain,HAS_SYMPTOM,A 44-year-old female patient presented to the ...
1,E_002,PMC10106591_01,physical examination,admission,SUPPORTS,upon
2,E_003,PMC10106591_01,pulmonary examination,normal heart,SUPPORTS,revealed
3,E_004,PMC10106591_01,electrocardiogram,ecg,SUPPORTS,(
4,E_005,PMC10106591_01,acute coronary syndrome,acs,SUPPORTS,(


Dados de casos carregados! Total de casos: 56


,article_id,age,case_id,case_text,gender
0,PMC5137649,44.0,PMC5137649_01,A 44-year-old woman presented with a 3-day his...,Female
1,PMC9387390,57.0,PMC9387390_01,A 57-year-old man with no significant past med...,Male
2,PMC3437073,55.0,PMC3437073_01,A 55-year-old male presented with a gradually ...,Male
3,PMC7102447,65.0,PMC7102447_01,A 65-year-old male with a past medical history...,Male
4,PMC6354154,30.0,PMC6354154_01,A 30-year-old nulligravida presented herself i...,Female


## Preparando Weak Supervision e Gold Standard

In [58]:
df_edges['contexto'] = df_edges['contexto'].fillna('[ADJACENTE]')

casos_forcar_treino = ['PMC5137649_01', 'PMC9387390_01', 'PMC3437073_01', 'PMC7102447_01']

# Lista de todos os casos disponíveis 
casos_unicos = df_casos['case_id'].unique()
casos_para_dividir = [c for c in casos_unicos if c not in casos_forcar_treino]

# Divisão
casos_treino_split, casos_teste = train_test_split(casos_para_dividir, test_size=0.05, random_state=43)

# Adiciona os casos de volta ao treino 
casos_treino = list(casos_treino_split) + casos_forcar_treino

# Treino continua usando as extrações ruidosas em massa do Projeto 1
df_treino_ruidoso = df_edges[df_edges['case_id'].isin(casos_treino)].copy()

# Teste será gerado apenas com os textos brutos para anotação do zero
df_export_casos = df_casos[df_casos['case_id'].isin(casos_teste)].copy()
df_export_casos = df_export_casos[['case_id', 'case_text']]

# Adicionando colunas vazias 
df_export_casos['source_label'] = ''
df_export_casos['target_label'] = ''
df_export_casos['relation'] = ''
df_export_casos.to_csv(PROCESSED / 'para_anotar_gold_standard.csv', index=False)

print(f"Casos para Treino (Ruído): {len(casos_treino)} casos -> {len(df_treino_ruidoso)} arestas.")
print(f"Casos para Teste (Gold Standard): {len(casos_teste)} casos brutos exportados para anotação manual.")

Casos para Treino (Ruído): 53 casos -> 620 arestas.
Casos para Teste (Gold Standard): 3 casos brutos exportados para anotação manual.


## TF-IDF e DataLoader PyTorch

In [59]:
#df_teste_gold = pd.read_csv(PROCESSED / 'gold_standard_anotado.csv')
df_teste_gold = pd.read_csv(PROCESSED / 'para_anotar_gold_standard.csv')

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=2000)
X_treino_tfidf = vectorizer.fit_transform(df_treino_ruidoso['contexto'])
#X_teste_tfidf = vectorizer.transform(df_teste_gold['contexto'])

labels_unicas = df_edges['relation'].unique()
label2id = {label: idx for idx, label in enumerate(labels_unicas)}
id2label = {idx: label for label, idx in label2id.items()}

class MLPDataset(Dataset):
    def __init__(self, matriz_tfidf, labels):
        self.x = torch.tensor(matriz_tfidf.toarray(), dtype=torch.float32)
        self.y = torch.tensor([label2id.get(l, 0) for l in labels], dtype=torch.long)
        
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

treino_dataset = MLPDataset(X_treino_tfidf, df_treino_ruidoso['relation'].values)
treino_loader = DataLoader(treino_dataset, batch_size=32, shuffle=True)

# NOTA IMPORTANTE: Quando o Excel for preenchido pela equipe, alterar 'relation' para a coluna manual e ler o arquivo novo!
#teste_dataset = MLPDataset(X_teste_tfidf, df_teste_gold['relation_corrigida_manualmente'].values)
#teste_loader = DataLoader(teste_dataset, batch_size=32, shuffle=False)

print("Tudo pronto! Pipeline de Machine Learning pode iniciar.")
batch_x, batch_y = next(iter(treino_loader))
print(f"Tensores de Entrada X: {batch_x.shape}")
print(f"Tensores de Label Y: {batch_y.shape}")

Tudo pronto! Pipeline de Machine Learning pode iniciar.
Tensores de Entrada X: torch.Size([32, 1926])
Tensores de Label Y: torch.Size([32])
